In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:

import os
import glob

print("=== KAGGLE INPUT DATASETS ===")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level <= 4:
        print(root)
        for f in files:
            print("   FILE:", f)

print("\n=== SEARCHING FOR ALL TASK FOLDERS ===")

for task in ["task1_lora", "task2_rag", "task3_local"]:
    matches = glob.glob(
        f"/kaggle/input/**/{task}",
        recursive=True
    )
    print(task, ":", matches)


=== KAGGLE INPUT DATASETS ===
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/kasiiiii
/kaggle/input/datasets/kasiiiii/tndrrr
/kaggle/input/datasets/kasiiiii/tndrrr/tndrr
   FILE: .gitignore
   FILE: INTERVIEW_PREP.md
   FILE: README.md
   FILE: REPORT_TEMPLATE.md
   FILE: requirements.txt

=== SEARCHING FOR ALL TASK FOLDERS ===
task1_lora : ['/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/task1_lora']
task2_rag : ['/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/task2_rag']
task3_local : ['/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/task3_local']


In [5]:

import os
import shutil
import subprocess

# Locate assignment files
SOURCE = "/kaggle/input/datasets/kasiiiii/tndrrr/tndrr/task2_rag"
WORK = "/kaggle/working/task2_rag"

assert os.path.isdir(SOURCE), "Task 2 folder not found!"

# Copy to writable workspace
if os.path.exists(WORK):
    shutil.rmtree(WORK)

shutil.copytree(SOURCE, WORK)
os.chdir(WORK)

print("TASK 2 FILES:", os.listdir("."))

# Install dependencies
subprocess.run([
    "pip", "install", "-q",
    "numpy", "faiss-cpu", "pypdf", "sentence-transformers"
], check=True)

# Download Llama 2 research paper
subprocess.run([
    "wget", "-q", "-L",
    "-O", "doc.pdf",
    "https://arxiv.org/pdf/2307.09288"
], check=True)

assert os.path.getsize("doc.pdf") > 100_000

# Copy question template
shutil.copy("questions_template.json", "questions.json")

print("\nPDF downloaded:", round(os.path.getsize("doc.pdf") / 1024 / 1024, 2), "MB")
print("TASK 2 SETUP SUCCESSFUL")


TASK 2 FILES: ['rag.py', 'questions_template.json']

PDF downloaded: 13.03 MB
TASK 2 SETUP SUCCESSFUL


In [6]:

import re
import json
from pypdf import PdfReader

os.chdir("/kaggle/working/task2_rag")

reader = PdfReader("doc.pdf")
pages = reader.pages

print("Total PDF pages:", len(pages))

all_sentences = []

for page_no, page in enumerate(pages, start=1):
    text = page.extract_text() or ""
    sentences = re.split(r'(?<=[.!?])\s+', text.replace("\n", " "))
    for sentence in sentences:
        sentence = " ".join(sentence.split())
        if len(sentence.split()) >= 8:
            all_sentences.append({
                "text": sentence,
                "page": page_no
            })

topics = [
    ("model sizes", ["7 billion", "70 billion"]),
    ("training tokens", ["2 trillion", "tokens"]),
    ("context length", ["4096", "context length"]),
    ("training data", ["publicly available", "data"]),
    ("architecture", ["transformer architecture"]),
    ("normalization", ["RMSNorm"]),
    ("activation", ["SwiGLU"]),
    ("position embeddings", ["rotary", "positional embeddings"]),
    ("attention", ["grouped-query attention"]),
    ("optimizer", ["AdamW"]),
    ("learning rate", ["cosine learning rate"]),
    ("fine tuning", ["supervised fine-tuning"]),
    ("human feedback", ["human feedback"]),
    ("reinforcement learning", ["reinforcement learning"]),
    ("safety", ["safety", "Llama 2-Chat"])
]

questions = []

for topic, keywords in topics:
    candidates = [
        s for s in all_sentences
        if all(k.lower() in s["text"].lower() for k in keywords)
    ]

    if not candidates:
        candidates = [
            s for s in all_sentences
            if keywords[0].lower() in s["text"].lower()
        ]

    if candidates:
        selected = candidates[0]
        questions.append({
            "q": f"What does the Llama 2 paper report about {topic}?",
            "gold": selected["text"],
            "page": selected["page"]
        })

print("Questions generated:", len(questions))

for i, q in enumerate(questions, 1):
    print(f"\n{i}. {q['q']}")
    print("Gold:", q["gold"])
    print("Page:", q["page"])

assert len(questions) == 15, "Check missing question topics"

with open("questions.json", "w") as f:
    json.dump(questions, f, indent=2)

print("\nSaved questions.json")


Total PDF pages: 77


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Questions generated: 15

1. What does the Llama 2 paper report about model sizes?
Gold: Llama 2: Open Foundation and Fine-Tuned Chat Models Hugo Touvron∗ Louis Martin† Kevin Stone† Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang A

In [7]:

import json
import os
from collections import Counter

os.chdir("/kaggle/working/task2_rag")

with open("questions.json", "r") as f:
    questions = json.load(f)

print("Total questions:", len(questions))
print("Unique questions:", len(set(q["q"] for q in questions)))
print("Unique gold passages:", len(set(q["gold"] for q in questions)))

for i, q in enumerate(questions, 1):
    print(f"\n{i}. {q['q']}")
    print("Page:", q["page"])
    print("Gold:", q["gold"][:250])


Total questions: 15
Unique questions: 15
Unique gold passages: 11

1. What does the Llama 2 paper report about model sizes?
Page: 1
Gold: Llama 2: Open Foundation and Fine-Tuned Chat Models Hugo Touvron∗ Louis Martin† Kevin Stone† Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Mo

2. What does the Llama 2 paper report about training tokens?
Page: 5
Gold: We trained on 2 trillion tokens of data as this provides a good performance–cost trade-off, up-sampling the most factual sources in an effort to increase knowledge and dampen hallucinations.

3. What does the Llama 2 paper report about context length?
Page: 54
Gold: We applied first GAtt toLlama 1, which was pretrained with a context length of 2048 tokens and then fine-tuned with 4096 max length.

4. What does the Llama 2 paper report about training data?
Page: 4
Gold: Llama 2, an updated version ofLlama 1, trained on a new mix o

In [9]:

!pip install -q PyMuPDF

import fitz
import os

os.chdir("/kaggle/working/task2_rag")

doc = fitz.open("doc.pdf")

print("PDF pages:", len(doc))
print("First page preview:")
print(doc[0].get_text()[:1000])

print("\nPyMuPDF installation successful!")


PDF pages: 77
First page preview:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗
Louis Martin†
Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodr

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [11]:

import pymupdf
import re
import os

os.chdir("/kaggle/working/task2_rag")
doc = pymupdf.open("doc.pdf")

keywords = [
    "7 billion", "70 billion", "RMSNorm",
    "SwiGLU", "rotary", "RoPE",
    "normalization", "activation"
]

for keyword in keywords:
    print("\n" + "="*65)
    print("SEARCH:", keyword)

    found = 0
    for page_no, page in enumerate(doc, 1):
        text = " ".join(page.get_text().split())
        for match in re.finditer(re.escape(keyword), text, re.I):
            start = max(0, match.start() - 250)
            end = min(len(text), match.end() + 350)
            print(f"Page {page_no}: {text[start:end]}")
            found += 1
            if found >= 2:
                break
        if found >= 2:
            break

    if not found:
        print("No matching text found.")



SEARCH: 7 billion
Page 1: ie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic Sergey Edunov Thomas Scialom∗ GenAI, Meta Abstract In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters. Our fine-tuned LLMs, called Llama 2-Chat, are optimized for dialogue use cases. Our models outperform open-source chat models on most benchmarks we tested, and based on our human evaluations for helpfulness and safety, may be a suitable substitute for closed- source models. We provide a detailed description of our approac
Page 36: 023). We are committed to continuing our work engaging with the broader policy, academic, and industry community on these issues. 7 Conclusion In this study, we have introduced Llama 2, a new family of pretrained and fine-tuned models with scales of 7 billion to 70 billion parameters. These models have demonstrated their competitiveness with existing o

In [12]:

import pymupdf
import re
import json
import os

os.chdir("/kaggle/working/task2_rag")
doc = pymupdf.open("doc.pdf")

# Extract actual sentences with page numbers
sentences = []

for page_no, page in enumerate(doc, 1):
    text = " ".join(page.get_text().split())
    for s in re.split(r'(?<=[.!?])\s+', text):
        s = s.strip()
        if 10 <= len(s.split()) <= 100:
            sentences.append({"text": s, "page": page_no})

# Select distinct evidence passages
specs = [
    ("model sizes", ["ranging in scale from 7 billion to 70 billion"]),
    ("training tokens", ["2 trillion tokens"]),
    ("pretraining data", ["publicly available data"]),
    ("transformer architecture", ["standard transformer architecture"]),
    ("normalization", ["RMSNorm"]),
    ("activation function", ["SwiGLU activation function"]),
    ("position embeddings", ["rotary positional embeddings"]),
    ("grouped query attention", ["grouped-query attention"]),
    ("optimizer", ["AdamW optimizer"]),
    ("learning rate schedule", ["cosine learning rate schedule"]),
    ("supervised fine tuning", ["application of supervised fine-tuning"]),
    ("human feedback", ["alignment with human preferences"]),
    ("reward modeling", ["iterative reward modeling data"]),
    ("safety improvements", ["safety improvements of Llama 2-Chat"]),
    ("context length", ["doubled the context length"])
]

questions = []
used = set()

for topic, keywords in specs:
    candidates = [
        x for x in sentences
        if all(k.lower() in x["text"].lower() for k in keywords)
        and x["text"] not in used
        and x["page"] != 2
    ]

    if not candidates:
        print("MISSING:", topic)
        continue

    selected = candidates[0]
    used.add(selected["text"])

    questions.append({
        "q": f"According to the Llama 2 paper, what is reported about {topic}?",
        "gold": selected["text"],
        "page": selected["page"]
    })

print("Total:", len(questions))
print("Unique passages:", len(used))

if len(questions) == 15 and len(used) == 15:
    with open("questions.json", "w") as f:
        json.dump(questions, f, indent=2)
    print("SUCCESS: 15 questions saved.")
else:
    print("STOP: Review missing topics before evaluation.")

for i, q in enumerate(questions, 1):
    print(f"{i}. {q['q']} | Page {q['page']}")


MISSING: model sizes
MISSING: normalization
MISSING: activation function
MISSING: position embeddings
Total: 11
Unique passages: 11
STOP: Review missing topics before evaluation.
1. According to the Llama 2 paper, what is reported about training tokens? | Page 5
2. According to the Llama 2 paper, what is reported about pretraining data? | Page 4
3. According to the Llama 2 paper, what is reported about transformer architecture? | Page 5
4. According to the Llama 2 paper, what is reported about grouped query attention? | Page 4
5. According to the Llama 2 paper, what is reported about optimizer? | Page 5
6. According to the Llama 2 paper, what is reported about learning rate schedule? | Page 5
7. According to the Llama 2 paper, what is reported about supervised fine tuning? | Page 5
8. According to the Llama 2 paper, what is reported about human feedback? | Page 3
9. According to the Llama 2 paper, what is reported about reward modeling? | Page 5
10. According to the Llama 2 paper, what

In [13]:
import pymupdf
import re

pdf = pymupdf.open("doc.pdf")

# Search the paper for distinct passages to fill the missing topics
searches = {
    "model sizes": ["7 billion to 70 billion"],
    "normalization": ["RMSNorm"],
    "activation function": ["SwiGLU"],
    "position embeddings": ["rotary positional embeddings"],
}

for topic, terms in searches.items():
    print(f"\n--- {topic} ---")
    found = False

    for page_no, page in enumerate(pdf):
        text = page.get_text()
        for term in terms:
            match = re.search(re.escape(term), text, re.IGNORECASE)
            if match:
                start = max(0, match.start() - 250)
                end = min(len(text), match.end() + 350)
                print(f"Page {page_no + 1}:")
                print(text[start:end].replace("\n", " "))
                found = True
                break
        if found:
            break

    if not found:
        print("Not found")


--- model sizes ---
Page 1:
ie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic Sergey Edunov Thomas Scialom∗ GenAI, Meta Abstract In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters. Our fine-tuned LLMs, called Llama 2-Chat, are optimized for dialogue use cases. Our models outperform open-source chat models on most benchmarks we tested, and based on our human evaluations for helpfulness and safety, may be a suitable substitute for closed- source models. We provide a detailed description of our approach to fine-tuni

--- normalization ---
Page 5:
of our models; results can be found in Section 4.1. 2.2 Training Details We adopt most of the pretraining setting and model architecture from Llama 1. We use the standard transformer architecture (Vaswani et al., 2017), apply pre-normalization using RMSNorm (Zhang and Sennrich, 2019), use the SwiGLU activation func

In [15]:
import os

os.chdir("/kaggle/working/task2_rag")

print("=" * 60)
print("QUESTION TEMPLATE")
print("=" * 60)

with open("questions_template.json", "r") as f:
    print(f.read())

print("\n" + "=" * 60)
print("RAG SCRIPT — FIRST 260 LINES")
print("=" * 60)

with open("rag.py", "r") as f:
    for i, line in enumerate(f):
        if i >= 260:
            break
        print(f"{i+1}: {line}", end="")

QUESTION TEMPLATE
[
  {"q": "WRITE YOUR OWN QUESTION 1", "gold": "Paste 1-2 verbatim sentences from the PDF that answer it", "page": null},
  {"q": "WRITE YOUR OWN QUESTION 2", "gold": "Paste 1-2 verbatim sentences from the PDF that answer it", "page": null}
]


RAG SCRIPT — FIRST 260 LINES
1: """RAG from scratch: sliding-window chunks -> bge-small embeddings -> FAISS (dense) + own BM25 -> RRF hybrid.
2: Metric: recall@k = fraction of questions where ANY top-k chunk contains the gold passage
3: (hit = >=70% of the gold passage's word trigrams appear in the chunk)."""
4: import argparse, json, re, os, numpy as np, faiss
5: from collections import Counter
6: from pypdf import PdfReader
7: from sentence_transformers import SentenceTransformer
8: 
9: QPREFIX = "Represent this sentence for searching relevant passages: "
10: toks = lambda s: re.findall(r"[a-z0-9]+", s.lower())
11: grams = lambda t, n=3: {tuple(t[i:i + n]) for i in range(len(t) - n + 1)}
12: 
13: def chunk(pdf, size, overlap)

In [16]:

import os, re, json
from pypdf import PdfReader

os.chdir("/kaggle/working/task2_rag")

PDF = "doc.pdf"
reader = PdfReader(PDF)

# Each entry uses a topic that can be found in the actual paper.
specs = [
    ("What range of parameter sizes does the Llama 2 model family cover?",
     "ranging in scale from 7 billion to 70 billion", 1, 1),

    ("What use cases were the fine-tuned Llama 2-Chat models optimized for?",
     "optimized for dialogue use cases", 1, 1),

    ("How did Llama 2 perform compared with open-source chat models?",
     "outperform open-source chat models", 1, 1),

    ("Which qualities were assessed in the human evaluations?",
     "human evaluations for helpfulness and safety", 1, 1),

    ("What new data mixture was used to train Llama 2?",
     "new mix of publicly available data", 4, 77),

    ("By what percentage was the pretraining corpus increased?",
     "increased the size of the pretraining corpus by 40%", 4, 77),

    ("What change was made to the model's context length?",
     "doubled the context length of the model", 4, 77),

    ("Which attention mechanism was adopted by Llama 2?",
     "grouped-query attention", 4, 77),

    ("How many tokens was Llama 2 pretrained on?",
     "2 trillion tokens", 5, 77),

    ("Which normalization, activation and positional embedding methods were used?",
     "RMSNorm", 5, 77),

    ("Which optimizer was used during training?",
     "AdamW optimizer", 5, 77),

    ("What learning rate schedule was used?",
     "cosine learning rate schedule", 5, 77),

    ("What supervised fine-tuning approach is described in the paper?",
     "supervised fine-tuning", 3, 77),

    ("What does the paper describe about reinforcement learning from human feedback?",
     "reinforcement learning with human feedback", 3, 77),

    ("What role does reward modeling play in the training approach?",
     "reward model", 3, 77),
]

# Extract text using the same PDF library as rag.py
pages = []
for page_no, page in enumerate(reader.pages, start=1):
    text = " ".join((page.extract_text() or "").split())
    if page_no == 1 and "Abstract" in text:
        text = text.split("Abstract", 1)[1]
    pages.append(text)

questions = []
missing = []

for question, keyword, first_page, last_page in specs:
    found = None

    for page_no in range(first_page, min(last_page, len(pages)) + 1):
        text = pages[page_no - 1]

        # Find the sentence containing the keyword
        sentences = re.split(r'(?<=[.!?])\s+', text)
        for sentence in sentences:
            sentence = sentence.strip()
            if keyword.lower() in sentence.lower():
                found = {
                    "q": question,
                    "gold": sentence,
                    "page": page_no
                }
                break

        if found:
            break

    if found:
        questions.append(found)
    else:
        missing.append((question, keyword))

# Validate against the exact text used by rag.py
full_text = " ".join(pages)
normalize = lambda s: " ".join(s.lower().split())

invalid = [
    q["q"] for q in questions
    if normalize(q["gold"]) not in normalize(full_text)
]

print("Questions found:", len(questions))
print("Missing:", missing)
print("Invalid gold passages:", invalid)

if len(questions) == 15 and not missing and not invalid:
    with open("questions.json", "w") as f:
        json.dump(questions, f, indent=2)

    print("\nSUCCESS: questions.json saved with 15 verified questions.")
    for i, q in enumerate(questions, 1):
        print(f'{i}. Page {q["page"]}: {q["q"]}')
else:
    print("\nSTOP: Validation failed. Do not run the benchmark.")


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Questions found: 15
Missing: []
Invalid gold passages: []

SUCCESS: questions.json saved with 15 verified questions.
1. Page 1: What range of parameter sizes does the Llama 2 model family cover?
2. Page 1: What use cases were the fine-tuned Llama 2-Chat models optimized for?
3. Page 1: How did Llama 2 perform compared with open-source chat models?
4. Page 1: Which qualities were assessed in the human evaluations?
5. Page 4: What new data mixture was used to train Llama 2?
6. Page 4: By what percentage was the pretraining corpus increased?
7. Page 4: What change was made to the model's context length?
8. Page 4: Which attention mechanism was adopted by Llama 2?
9. Page 5: How many tokens was Llama 2 pretrained on?
10. Page 5: Which normalization, activation and positional embedding methods were used?
11. Page 5: Which optimizer was used during training?
12. Page 5: What learning rate schedule was used?
13. Page 5: What supervised fine-tuning approach is described in the paper?
14. Page 

In [17]:
%cd /kaggle/working/task2_rag
!python rag.py --pdf doc.pdf --questions questions.json --sizes 128,384

/kaggle/working
15 questions (need 15)
config_sentence_transformers.json: 100%|████████| 124/124 [00:00<00:00, 303kB/s]
README.md: 100%|███████████████████████████| 94.8k/94.8k [00:00<00:00, 74.3MB/s]
sentence_bert_config.json: 100%|██████████████| 52.0/52.0 [00:00<00:00, 138kB/s]
config.json: 100%|█████████████████████████████| 743/743 [00:00<00:00, 2.18MB/s]

model.safetensors: downloading bytes: █████████████▍        | 81.5MB, 6.00MB/s  
model.safetensors: downloading bytes: ██████████████████████| 86.7MB, 8.35MB/s  
model.safetensors: reconstructing file: 100%|██████|  133MB /  133MB, 13.1MB/s  
tokenizer_config.json: 100%|████████████████████| 366/366 [00:00<00:00, 960kB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 20.4MB/s]
tokenizer.json: 100%|████████████████████████| 711k/711k [00:00<00:00, 51.3MB/s]
config.json: 100%|██████████████████████████████| 190/190 [00:00<00:00, 517kB/s]
Exceeded 5000 form XObject invocations while extracting text; further

In [18]:
import os
import json

os.chdir("/kaggle/working/task2_rag")

print("Recall results:")
print(open("results/recall.json").read())

print("\nGenerated files:")
for root, dirs, files in os.walk("results"):
    for f in files:
        print(os.path.join(root, f))

Recall results:
{
 "128": {
  "n_chunks": 383,
  "recall@5_dense": 0.267,
  "recall@5_bm25": 0.4,
  "recall@5_hybrid": 0.4
 },
 "384": {
  "n_chunks": 128,
  "recall@5_dense": 0.533,
  "recall@5_bm25": 0.533,
  "recall@5_hybrid": 0.533
 }
}

Generated files:
results/failures_128.json
results/recall.json
results/failures_384.json
